# EYEWAZ — train your Urdu Piper voice (Kaggle or Colab)

**KAGGLE (recommended):** new notebook → right panel → **Accelerator: GPU T4 x2**,
**Internet: On**. Then **Add Input → Upload** `dataset-male.zip` (creates a dataset
under /kaggle/input). Run all.

**COLAB:** Runtime → GPU; put `dataset-male.zip` in Google Drive/eyewaz/. Run all.

⚠️ Keep the tab open. On Kaggle, also tick **Save Version → keep session** for long runs.


### 1. Detect platform + find your dataset


In [ ]:
import os, glob
KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if KAGGLE else '/content'
VOICE = 'male'
os.chdir(WORK)
if not KAGGLE:
    from google.colab import drive; drive.mount('/content/drive')
cands = (glob.glob('/kaggle/input/**/dataset-*.zip', recursive=True)
         + glob.glob('/content/drive/MyDrive/eyewaz/dataset-*.zip')
         + glob.glob(WORK+'/dataset-*.zip'))
assert cands, 'Upload dataset-male.zip (Kaggle: Add Input>Upload; Colab: Drive/eyewaz/)'
ZIP = cands[0]; print('platform:', 'Kaggle' if KAGGLE else 'Colab', '| dataset:', ZIP)


### 2. GPU on?


In [ ]:
!nvidia-smi -L || print('NO GPU — enable a GPU accelerator first!')


### 3. Install piper1-gpl (compiles espeakbridge, keeps the vits trainer) — ~5 min


In [ ]:
import os, glob, shutil, subprocess
!apt-get -qq install -y espeak-ng libespeak-ng-dev build-essential cmake ninja-build >/dev/null 2>&1
if not os.path.exists(WORK+'/piper1'):
    !git clone -q https://github.com/OHF-Voice/piper1-gpl {WORK}/piper1
os.chdir(WORK+'/piper1')
# 1) non-editable build compiles espeakbridge*.so into the install dir
!pip install -q . 2>&1 | tail -4
pdir = subprocess.run(['python3','-c','import piper,os;print(os.path.dirname(piper.__file__))'],
                      capture_output=True, text=True).stdout.strip()
so = glob.glob(os.path.join(pdir,'espeakbridge*.so'))
assert so, f'espeakbridge did not compile (looked in {pdir})'
shutil.copy(so[0], WORK+'/eb.so')   # back up before editable reinstall removes it
# 2) editable install exposes piper.train.vits from source
!pip install -q -e . --force-reinstall --no-deps
# 3) drop the compiled bridge into the editable source
shutil.copy(WORK+'/eb.so', WORK+'/piper1/src/piper/'+os.path.basename(so[0]))
# 4) build the Cython alignment module
!bash build_monotonic_align.sh
!python3 -c "from piper import espeakbridge; from piper.train.vits.dataset import VitsDataModule; print('imports OK')"


### 4. Unzip dataset + pretrained checkpoint


In [ ]:
import os
os.chdir(WORK)
!unzip -oq {ZIP} -d {WORK}/
DATA = f'{WORK}/dataset-{VOICE}'
with open(f'{DATA}/metadata.csv',encoding='utf-8') as f, open(f'{WORK}/{VOICE}.csv','w',encoding='utf-8') as o:
    for line in f:
        line=line.strip()
        if '|' in line:
            cid,text=line.split('|',1); o.write(f'{cid}.wav|{text}\n')
print('clips:', len(os.listdir(f'{DATA}/wav')))
PT = f'{WORK}/pretrained.ckpt'
if not os.path.exists(PT):
    !wget -q -O {PT} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
print('pretrained MB:', round(os.path.getsize(PT)/1e6,1))


### 5. Train
**Watch live `Epoch N … loss=…`.** Leave it 30+ min. Checkpoints save to WORK/out.
Stop with ■ when trained enough, then run cell 6.


In [ ]:
import torch, os, glob
OUT = f'{WORK}/out'
latest = sorted(glob.glob(f'{OUT}/**/*.ckpt', recursive=True), key=os.path.getmtime)
if latest:
    RESUME = latest[-1]; print('resuming from', RESUME)
else:
    ck = torch.load(f'{WORK}/pretrained.ckpt', map_location='cpu', weights_only=False)
    for k in ['hyper_parameters','hparams_name','datamodule_hyper_parameters','loops']: ck.pop(k, None)
    ck['epoch']=0; ck['global_step']=0
    for s in (ck.get('lr_schedulers') or []):
        if isinstance(s, dict): s['last_epoch']=0; s['_step_count']=1
    torch.save(ck, f'{WORK}/warm.ckpt'); RESUME=f'{WORK}/warm.ckpt'; print('warm-start from Lessac')
os.chdir(WORK+'/piper1')
open(WORK+'/piper1/sitecustomize.py','w').write("import torch\n_o=torch.load\ndef _p(*a,**k):\n k['weights_only']=False\n return _o(*a,**k)\ntorch.load=_p\n")
!PYTHONPATH={WORK}/piper1 python3 -m piper.train fit \
  --data.voice_name eyewaz-urdu-{VOICE} \
  --data.csv_path {WORK}/{VOICE}.csv \
  --data.audio_dir {WORK}/dataset-{VOICE}/wav \
  --model.sample_rate 22050 \
  --data.espeak_voice ur \
  --data.cache_dir {WORK}/cache \
  --data.config_path {WORK}/eyewaz-urdu-{VOICE}.json \
  --data.batch_size 16 \
  --ckpt_path {RESUME} \
  --trainer.max_epochs 1000 \
  --trainer.default_root_dir {OUT} \
  --trainer.accelerator gpu --trainer.devices 1


### 6. Export to ONNX


In [ ]:
import glob, os
cks = sorted(glob.glob(f'{WORK}/out/**/*.ckpt', recursive=True), key=os.path.getmtime)
assert cks, 'No checkpoint yet — let cell 5 run more epochs.'
ck = cks[-1]; print('exporting', ck)
!python3 -m piper.train.export_onnx --checkpoint {ck} --output-file {WORK}/eyewaz-urdu-{VOICE}.onnx
!cp {WORK}/eyewaz-urdu-{VOICE}.json {WORK}/eyewaz-urdu-{VOICE}.onnx.json
print('Saved to', WORK, '- download eyewaz-urdu-male.onnx + .onnx.json')
if not KAGGLE:
    from google.colab import files
    files.download(f'{WORK}/eyewaz-urdu-{VOICE}.onnx'); files.download(f'{WORK}/eyewaz-urdu-{VOICE}.onnx.json')


### Done
**Kaggle:** download the two files from the **Output** panel (right side).
Send me `eyewaz-urdu-male.onnx` + `.onnx.json` and I wire your voice into every EYEWAZ surface.
